# Parsiranje scenarija: *Kockar* (*The Card Counter*, 2021)

Izdvajanje iskaza Vilijama Tela iz scenarija za film *Kockar*. Ovo je jedan od četiri parsera urađena za master rad *Božji usamljeni ljudi: računarska analiza jezika protagonista u scenarijima Pola Šrejdera* (Univerzitet Singidunum, 2026). Svaki scenario u korpusu ima svoj parser, jer se četiri scenarija razlikuju po načinu na koji su zapisani ime lika, parentetika, dijalog i didaskalije. Postupak je opisan u potpoglavljima 5.2 i 5.3 rada.

**Izvor.** Scenario je preuzet sa sajta scriptslug.com (tačna adresa navedena je u literaturi rada) kao PDF od 91 strane. Reč je o revidiranoj verziji za snimanje od 17. februara 2020, u kojoj su zadržani i raniji revizioni slojevi, obeleženi zaglavljima i revizionim zvezdicama. Fajl nije deo repozitorijuma, jer je scenario zaštićen autorskim pravom. Za ponavljanje postupka PDF treba sačuvati kao 'The-Card-Counter-Script.pdf', u istom folderu kao notebook.

**Izlaz.** 'tell_lines.csv', sa kolonama 'speaker', 'type', 'record_no' i 'line': 289 iskaza, od toga 251 dijalog i 38 naracije. Ovaj fajl je, zajedno sa izlazima ostala tri parsera, ulaz za notebook 'Analiza_scenarija.ipynb'.

**Pokretanje.** Redom, od prve ćelije. Potrebni paketi: 'pdfplumber' i 'pandas'. 

In [1]:
import pdfplumber
import re
import pandas as pd
from collections import defaultdict
from collections import Counter

## 1. Učitavanje scenarija

PDF se čita strana po strana pomoću biblioteke pdfplumber, koja za svaku reč daje njen položaj na strani. Reči se zatim slažu u redove, a za svaki red beleže se strana, x-koordinata prve reči i tekst.

Ovaj PDF traži tri prilagođavanja. Prvo, neki elementi su podebljani tako što je isti tekst odštampan dvaput, jedan preko drugog, pa bi se bez ispravke umesto 'TELL' dobilo 'TTEELLLL'. Duplirana slova uklanja 'dedupe_chars()'. Drugo, oznaka '(CONT'D)' ponegde stoji poen niže od imena lika, pa bi se pri grupisanju po tačnoj vertikalnoj poziciji našla u zasebnom redu. Zato se u isti red stavljaju reči čija se vertikalna pozicija razlikuje za najviše tri poena. Treće, izmenjeni redovi su na kraju obeleženi revizionim zvezdicama, koje se uklanjaju.

Uz to se "pametni" apostrofi i navodnici prevode u obične, kako bi ista reč u celom korpusu bila zapisana na isti način.

In [2]:
PDF_PATH = 'The-Card-Counter-Script.pdf'

def normalizuj(t):
    # pretvaranje apostrofa i navodnika u obične
    t = (t.replace('’', "'").replace('‘', "'")
            .replace('“', '"').replace('”', '"'))
    return re.sub(r'\s*\*+$', '', t) # uklanjanje revizionih zvezdica

pdf = pdfplumber.open(PDF_PATH)
all_rows = []

for pi, page in enumerate(pdf.pages):
    # Uklanjanje dupliranih slova (TTEELLLL u TELL)
    words = page.dedupe_chars().extract_words()
    # Reči u isti red ako im se y razlikuje za najviše 3 poena, jer (CONT'D) ponegde stoji poen niže od imena
    words = sorted(words, key=lambda w: w['top'])
    rows = []
    for w in words:
        if rows and abs(w['top'] - rows[-1][0]['top']) <= 3:
            rows[-1].append(w)
        else:
            rows.append([w])
    for ws in rows:
        ws = sorted(ws, key=lambda w: w['x0'])
        x0 = round(ws[0]['x0'])
        text = normalizuj(' '.join(w['text'] for w in ws))
        all_rows.append({'page': pi, 'x0': x0, 'text': text})


## 2. Prepoznavanje elemenata scenarija

Kao i u parseru za *Iskušenika*, elementi scenarija razlikuju se kombinacijom položaja i oblika reda. Položaj odvaja samo krajeve strane: sve što počinje levo od x = 140 je didaskalija ili zaglavlje scene (x ≈ 100, odnosno 46), a sve desno od x = 380 je prelaz, kao 'FADE TO BLACK.' na x ≈ 405. U srednjem pojasu tip reda određuje njegov oblik. Red koji počinje zagradom je parentetika, red koji odgovara obrascu imena lika je ime, a sve ostalo je dijalog.

Ime lika je zapisano velikim slovima, može sadržati cifre (jedan lik nosi oznaku '911') i ne završava se tačkom. Uz ime mogu stajati sufiksi 'V.O', 'CONT'D', 'O.S.' i 'O.C.'. Za razliku od ostala tri scenarija, ovde se oznaka naracije piše bez zagrada, kao 'TELL V.O.', pa obrazac dozvoljava sufikse i sa zagradama i bez njih.

Zasebnim obrascem prepoznaju se zaglavlja revizija na vrhu strana, kao 'Pink Rev. (2/13/2020) 22.', i brojevi strana, kao '22.' ili '13A.', gde slovo označava naknadno umetnutu stranu. Oni se u parsiranju preskaču.

In [3]:
# Scenario ima zaglavlje revizije revizije ("Pink Rev. (2/13/2020) 22.") ili samo broj strane ("22.", "13A.")
zaglavlje = re.compile(r"^(\w+ Rev\. \(\d+/\d+/\d+\)\s*)?\d+[A-Z]?\.$")

# ime: velika slova i cifre (lik 911), bez tačke na kraju;
# sufiksi V.O., CONT'D, O.S., O.C., sa zagradama ili bez njih (TELL V.O.)
speaker_name = re.compile(
    r"^([A-Z0-9][A-Z0-9 .'\-]{1,28}?)(?<!\.)"
    r"((?:\s*\(?(?:V\.O\.?|CONT'?D|O\.S\.?|O\.C\.?)\)?)*)\s*$"
)

def classify(x0, txt):
    if x0 < 140: return 'action' # didaskalije (100) i zaglavlja scena (46)
    if x0 > 380: return 'other'  # prelazi, npr. FADE TO BLACK. (405)
    if txt.startswith('('): return 'paren'
    if speaker_name.match(txt): return 'speaker'
    return 'dialogue'

## 3. Parsiranje

Parser prolazi kroz redove redom i od njih sklapa iskaze. Zaglavlja revizija i brojevi strana se preskaču. Ime lika otvara nov iskaz, redovi dijaloga dodaju se tekućem iskazu, a didaskalija ili prelaz ga zatvara. Pri zatvaranju se redovi spajaju u jedan tekst, a višestruki razmaci svode na jedan. Ako uz ime stoji sufiks 'V.O.', ceo iskaz se označava kao naracija.

Parentetika ne ulazi u tekst iskaza i ne prekida ga, pa se delovi replike pre i posle nje spajaju u jedan iskaz. Kad parentetika prelazi u sledeći red, preskaču se svi redovi do onog u kojem se zagrada zatvara.

Replika prelomljena preko strane obeležena je oznakom '(MORE)' na dnu strane i ponovljenim imenom lika sa '(CONT'D)' na vrhu sledeće. U tom slučaju parser ne otvara nov iskaz, nego nastavlja tekući, pa prelom strane ne deli repliku na dva iskaza. U ovom scenariju takvih preloma ima 18. Zbog toga je važno da se zaglavlja i brojevi strana uklone pre parsiranja: broj strane između '(MORE)' i '(CONT'D)' bi inače zatvorio iskaz. Oznaka '(CONT'D)' bez prethodnog '(MORE)', na primer kad lik nastavlja da govori posle didaskalije, otvara nov iskaz, kao i svako drugo ime.

In [4]:
records = []
current, current_vo, buf = None, False, []
more, in_paren = False, False

def flush():
    'Zatvori tekuću repliku i upiši je'
    global buf
    if current and buf:
        txt = re.sub(r'\s+', ' ', ' '.join(buf).strip())
        if txt:
            records.append((current, current_vo, txt))
    buf = []

for r in all_rows:
    txt = r['text'].strip()

    # Preskakanje praznih redova, zaglavlja revizija i brojeve strana
    if not txt or zaglavlje.match(txt):
        continue
    # nastavak parentetike koja prelazi u sledeći red
    if in_paren:
        if')' in txt:
            in_paren = False
        continue

    cls = classify(r['x0'], txt)

    if cls == 'speaker':
        m = speaker_name.match(txt)
        ime, sufiksi = m.group(1).strip(), m.group(2)
        # nastavak replike posle preloma strane: (MORE) na dnu, IME (CONT'D) na vrhu sledeće
        if more and ime == current and 'CONT' in sufiksi:
            more = False
            continue
        flush()
        more = False
        current = ime
        current_vo = 'V.O.' in sufiksi
    elif cls == 'paren':
        if txt == '(MORE)':
            more = True
        elif ')' not in txt:
            in_paren = True
        continue
    elif cls == 'dialogue':
        if current:
            buf.append(txt)
    else:
        flush()
        current = None
        more = False
flush()

### Kontrola: oblici parentetike

Pregled parentetika u scenariju po učestalosti. U ispisu nema oznake naracije u obliku parentetike, jer se naracija u ovom scenariju označava sufiksom uz ime lika, a '(MORE)' se javlja 18 puta. Redovi poput '(waits till Waitress' prvi su redovi parentetika koje prelaze u
sledeći red.

Jedan red u ispisu, '(false positives) and falling to', nije parentetika, nego red dijaloga koji počinje zagradom, pa ga parser preskače.

In [5]:
# kontrola: oblici parentetike i broj preloma (MORE)
paren =  [r['text'] for r in all_rows if classify(r['x0'], r['text']) == 'paren']
print(Counter(paren).most_common(30))

[('(MORE)', 18), ('(beat)', 18), ('(pause)', 9), ('(no response)', 3), ('(Cirk nods)', 3), ('(to Tell)', 3), ('(he shrugs)', 2), ('(smiles)', 2), ('(to Cirk)', 2), ('(shrugs)', 2), ('(turns over card)', 2), ('(gives her $100 bill)', 1), ('(they sit)', 1), ('(checks his watch)', 1), ('(he nods)', 1), ('(false positives) and failing to', 1), ('(Tell waits)', 1), ('(incredulous)', 1), ('(hard)', 1), ('(on phone)', 1), ('(extends hand)', 1), ('(taking the hint)', 1), ('(looks her in the eyes)', 1), ('(turns over those cards)', 1), ('(mimics it)', 1), ('(collects cards)', 1), ('(waits till Waitress', 1), ("(Cirk doesn't respond)", 1), ('(makes clicking sound)', 1), ('(nods)', 1)]


## 4. Izdvajanje Telovih iskaza i čuvanje

Svi iskazi se najpre smeštaju u tabelu i dobijaju redni broj u celom scenariju ('record_no'). Broj se dodeljuje pre izdvajanja protagoniste, pa obuhvata replike svih likova, dok didaskalije broj ne dobijaju. Zato dva Telova iskaza sa uzastopnim brojevima nije razdvojila replika drugog lika, nego najviše didaskalija. Na tome počiva spajanje iskaza naracije u blokove u glavnom notebooku.

Jedna Telova replika, na 71. strani, u scenariju je greškom označena kao 'COUNT DOWN'. Iz naizmeničnog dijaloga Tela i Cirka jasno je da je izgovara Tel, pa se ta oznaka preslikava na 'TELL'. To je jedini slučaj u korpusu u kojem je pripisivanje replike ispravljeno ručno.

Pre izdvajanja se ispisuje spisak likova po broju replika. Sufiksi poput 'V.O.' i 'CONT'D' ne ulaze u ime, pa se Tel u spisku vodi pod jednim imenom, 'TELL', bez obzira na mod. Oznaka 'ALL THREE TOGETHER', na 43. strani, stoji uz jednu reč koju Tel izgovara zajedno sa Cirkom i La Lindom i nije uračunata kao Telova, jer nije njegova pojedinačna replika. Oznaka 'THE CARD COUNTER' je naslov sa naslovne strane. Po obliku odgovara imenu lika, pa je red "Written by Paul Schrader" ispod njega zabeležen kao replika. To je prvi zapis u tabeli i ne utiče na Telov korpus.

Jedan kratak dnevnički zapis, na 18. strani, citiran je unutar didaskalije, a ne kao replika sa oznakom 'V.O.', pa nije ušao u korpus.

U koloni 'type' vrednost 'dialogue' označava dijalog, a 'voice_over' ono što se u radu naziva naracijom. Očekivani rezultat je 289 iskaza: 288 označenih kao 'TELL' i jedan označen kao 'COUNT DOWN', od toga 251 dijalog i 38 naracije.

In [6]:
df = pd.DataFrame(records, columns=['speaker_raw', 'is_vo', 'line'])
df['record_no'] = range(1, len(df) + 1) # redni broj u celom scenariju, potreban zbog kasnijeg spajanja blokova
# Jedna Tellova replika je u scenariju greškom označena kao COUNT DOWN
alijasi = {'COUNT DOWN': 'TELL'}
df['speaker'] = df['speaker_raw'].replace(alijasi)

print(sorted({s for s in df['speaker_raw'].unique() if 'V.O' in s.upper()}))
print(df['speaker_raw'].value_counts().head(20))

tell = df[df['speaker'] == 'TELL'].copy().reset_index(drop=True)
tell['type'] = tell['is_vo'].apply(lambda v: 'voice_over'  if v else 'dialogue')
tell = tell[['speaker', 'type', 'record_no' ,'line']]

print('Tellovih replika:', len(tell))
print(tell['type'].value_counts())

tell.to_csv('tell_lines.csv', index=False)

[]
speaker_raw
TELL                  288
CIRK                  116
LA LINDA              109
GORDO                  19
SLIPPERY JOE           16
HOSKINS                 8
JUDY ON SCREEN          5
DESK CLERK              4
MINNESOTA               4
SARA                    4
CIRK ON SCREEN          4
PA ANNOUNCER            3
CHARLIE                 2
911                     2
THE CARD COUNTER        1
TOURNAMENT CLERK        1
ALL THREE TOGETHER      1
SECOND PLAYER           1
RONAN                   1
COUNT DOWN              1
Name: count, dtype: int64
Tellovih replika: 289
type
dialogue      251
voice_over     38
Name: count, dtype: int64
